# EoMT trên BTXRD: phát hiện + phân đoạn u xương và phân loại lành / ác

Huấn luyện **EoMT** (Encoder-only Mask Transformer, CVPR 2025) của dự án trên bộ X-quang u xương **BTXRD**: một lần forward cho mỗi ảnh vừa **phát hiện** có u hay không, vừa **phân đoạn** vùng u, vừa **phân loại** u lành / ác.

- Ảnh có u: mỗi polygon là một mask, lớp `benign` hoặc `malignant`. Ảnh bình thường: 0 mask, mọi query học dự đoán lớp "no object" có sẵn của EoMT.
- Dữ liệu: dataset Kaggle [`nguyetanhmai/btxrd-full-split`](https://www.kaggle.com/datasets/nguyetanhmai/btxrd-full-split), 3.726 ảnh chia train : val : test = 80 : 15 : 5 (2.980 / 559 / 187).
- Notebook chỉ cấu hình và gọi code của dự án: `tools/prepare_btxrd.py` (tải, chia, thu nhỏ ảnh), `main.py fit` / `main.py test` (huấn luyện, đánh giá), `tools/export_weights.py`, `tools/predict_btxrd.py` (inference + đánh giá theo ảnh).

**Chạy trên Google Colab:** *Runtime → Change runtime type → GPU*, rồi *Run all*.
- Dataset Kaggle private: thêm `KAGGLE_USERNAME`, `KAGGLE_KEY` vào 🔑 *Secrets* của Colab (bật *Notebook access*).
- Kết quả lưu ở `MyDrive/btxrd_eomt/`. Colab ngắt giữa chừng thì *Run all* lại: huấn luyện tiếp từ `last.ckpt`.

## 1. Cấu hình

In [ ]:
import os
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = Path("/kaggle/input").exists()

# --- Paths ---
SAVE_TO_DRIVE = True  # Colab: keep results on Google Drive (survive a disconnect)
if ON_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
WORK_DIR = Path("/content") if ON_COLAB else Path("/kaggle/working") if ON_KAGGLE else Path.cwd().parent
OUT_DIR = Path("/content/drive/MyDrive/btxrd_eomt") if ON_COLAB and SAVE_TO_DRIVE else WORK_DIR / "btxrd_eomt"
REPO_URL = "https://github.com/67htb5vn/eomt.git"
REPO_BRANCH = "master"
REPO_DIR = WORK_DIR / "eomt" if (ON_COLAB or ON_KAGGLE) else Path.cwd()

# --- Data ---
KAGGLE_DATASET = "nguyetanhmai/btxrd-full-split"
DATA_SOURCE = "--src /kaggle/input/btxrd-full-split" if ON_KAGGLE else f"--kaggle-dataset {KAGGLE_DATASET}"
DATA_DIR = WORK_DIR / "btxrd_cache"  # resized copy written by tools/prepare_btxrd.py

# --- Model / training ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"  # EoMT-B (DINOv2), 640x640
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"  # fine-tune from EoMT-B COCO panoptic (README)
RUN_NAME = "btxrd_instance_eomt_base_640"
EPOCHS = 40
BATCH_SIZE_PER_GPU = 4  # fits a 16 GB T4
NUM_WORKERS = min(4, os.cpu_count() or 2)
LR = 1e-4
TUMOR_SCORE_THRESH = 0.5  # image has a tumor if the best query's tumor score is above this
RESUME = True  # continue from last.ckpt if it exists
SMOKE_TEST = False  # True = 2 train + 2 val batches, to check the pipeline

CKPT_DIR = OUT_DIR / "checkpoints"
RUN_CONFIG = OUT_DIR / "run.yaml"  # overrides of BASE_CONFIG for this run
TEST_CONFIG = OUT_DIR / "test.yaml"  # overrides for evaluation
BEST_WEIGHTS = OUT_DIR / "best.bin"
RESULTS_DIR = OUT_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Cài đặt: mã nguồn dự án và thư viện

In [ ]:
if not (REPO_DIR / "main.py").exists():
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements_btxrd.txt

import torch

NUM_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])

## 3. Chuẩn bị dữ liệu

`tools/prepare_btxrd.py` tải dataset từ Kaggle, chia 80:15:5 nếu dữ liệu chưa chia, và ghi một bản ảnh đã thu nhỏ (cạnh dài ≤ 1024 px, polygon co theo) vào `DATA_DIR`. Chạy lại sẽ bỏ qua nếu đã chuẩn bị xong.

In [ ]:
!python tools/prepare_btxrd.py {DATA_SOURCE} --out {DATA_DIR}

## 4. Huấn luyện

Lệnh của dự án: `main.py fit -c <config gốc> -c <config lần chạy>`. File `run.yaml` chỉ ghi đè những gì thay đổi theo máy: đường dẫn, số GPU, batch, số epoch, logger, nơi lưu checkpoint, và checkpoint COCO để fine-tune (`ckpt_path`, `load_ckpt_class_head: false` như README). Lịch mask annealing được tự tính theo tổng số bước.

In [ ]:
import yaml
from huggingface_hub import hf_hub_download

run_cfg = {
    "seed_everything": 0,
    "trainer": {
        "max_epochs": 1 if SMOKE_TEST else EPOCHS,
        "devices": max(1, NUM_GPUS),
        "accelerator": "gpu" if NUM_GPUS else "cpu",
        "precision": "16-mixed" if NUM_GPUS else "32-true",
        "strategy": "ddp" if NUM_GPUS > 1 else "auto",
        "log_every_n_steps": 50,
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": RUN_NAME},
        },
        "callbacks": [
            {
                "class_path": "lightning.pytorch.callbacks.ModelCheckpoint",
                "init_args": {
                    "dirpath": str(CKPT_DIR),
                    "monitor": "metrics/val_score",  # (detection AUROC + Dice + benign/malignant AUROC) / 3
                    "mode": "max",
                    "save_last": True,
                    "save_top_k": 1,
                    "filename": "best-epoch{epoch:02d}",
                    "auto_insert_metric_name": False,
                },
            }
        ],
    },
    "model": {
        "init_args": {
            "lr": LR,
            "tumor_score_thresh": TUMOR_SCORE_THRESH,
            "ckpt_path": hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin"),
            "load_ckpt_class_head": False,  # 133 COCO classes -> 2 BTXRD classes
        }
    },
    "data": {"init_args": {"path": str(DATA_DIR), "batch_size": BATCH_SIZE_PER_GPU, "num_workers": NUM_WORKERS}},
}
if SMOKE_TEST:
    run_cfg["trainer"].update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))

last_ckpt = CKPT_DIR / "last.ckpt"
resume = f"--ckpt_path {last_ckpt}" if RESUME and not SMOKE_TEST and last_ckpt.exists() else ""
!python main.py fit -c {BASE_CONFIG} -c {RUN_CONFIG} --compile_disabled {resume}

## 5. Xuất trọng số tốt nhất

Chuyển checkpoint tốt nhất (theo `metrics/val_score`) thành state dict `.bin`, định dạng README dùng cho `--model.ckpt_path`.

In [ ]:
!python tools/export_weights.py --ckpt-dir {CKPT_DIR} --out {BEST_WEIGHTS}

## 6. Đánh giá trên tập test (CLI của dự án)

Như phần *Evaluating* của README: tắt masked attention khi suy luận và nạp trọng số qua `model.ckpt_path`, chạy trên 1 GPU. In ra mask AP và các chỉ số phát hiện / phân đoạn / lành-ác / 3 lớp.

In [ ]:
test_cfg = {
    "trainer": {
        "devices": 1,
        "strategy": "auto",
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": f"{RUN_NAME}_test"},
        },
    },
    "model": {
        "init_args": {
            "ckpt_path": str(BEST_WEIGHTS),
            "load_ckpt_class_head": True,
            "network": {"init_args": {"masked_attn_enabled": False}},
        }
    },
}
TEST_CONFIG.write_text(yaml.safe_dump(test_cfg, sort_keys=False))
!python main.py test -c {BASE_CONFIG} -c {RUN_CONFIG} -c {TEST_CONFIG}

## 7. Inference và đánh giá theo từng ảnh trên tập test

`tools/predict_btxrd.py` dự đoán từng ảnh (có u hay không, vùng u, lành / ác), ghi `predictions.csv`, báo cáo phân loại cho từng phần (`report.txt`, `metrics.json`), confusion matrix, ROC và hình minh họa: ảnh ngẫu nhiên, u bị bỏ sót, báo động nhầm, nhầm lành / ác, Dice thấp.

In [ ]:
from IPython.display import Image, display

TEST_RESULTS = RESULTS_DIR / "test"
!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} --weights {BEST_WEIGHTS} --data-path {DATA_DIR} --split test --out {TEST_RESULTS}

display(Image(str(TEST_RESULTS / "confusion_matrix.png")), Image(str(TEST_RESULTS / "roc.png")))
for p in sorted((TEST_RESULTS / "vis").glob("*.png")):
    display(Image(str(p)))

## 8. Dự đoán cho ảnh X-quang mới

Thay `NEW_IMAGES` bằng đường dẫn ảnh (hoặc thư mục ảnh) của bạn.

In [ ]:
NEW_IMAGES = sorted((DATA_DIR / "test" / "images").iterdir())[:3]  # example: 3 test images
NEW_RESULTS = RESULTS_DIR / "new"
image_args = " ".join(map(str, NEW_IMAGES))
!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} --weights {BEST_WEIGHTS} --images {image_args} --out {NEW_RESULTS}

for p in sorted((NEW_RESULTS / "vis").glob("*.png")):
    display(Image(str(p)))

## Kết quả đầu ra (trong `OUT_DIR`)

| File | Nội dung |
|---|---|
| `best.bin` | Trọng số tốt nhất (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/*.ckpt` | Checkpoint Lightning (best + last, dùng để resume) |
| `run.yaml`, `test.yaml` | Phần ghi đè cấu hình của lần chạy / lần đánh giá |
| `logs/` | Log CSV của huấn luyện và đánh giá |
| `results/test/` | `predictions.csv`, `report.txt`, `metrics.json`, `confusion_matrix.png`, `roc.png`, `vis/` |
| `results/new/` | Dự đoán cho ảnh mới |